# CorpBrain — Notebook 1: Whisper Transcription Benchmark

**Best run on Kaggle GPU (T4) for speed comparison.**
Also works locally on CPU (just slower).

**What this does:**
- Tests `faster-whisper` (what the cognitive service uses)
- Generates a realistic meeting audio file
- Transcribes it and measures speed (Real-Time Factor)
- Shows word-level timestamps

> 🎯 Goal: Confirm the transcription component works before plugging into the full pipeline.

## Step 1 — Install faster-whisper

In [1]:
# ─── Install faster-whisper ──────────────────────────────────────────────────
# NOTE: Requires "Internet ON" in Kaggle Settings (right panel → Internet → On)
# The Whisper model weights (~140MB) will be downloaded from HuggingFace.
!pip install faster-whisper -q
print("faster-whisper installed ✅")
print()
print("⚠️  If you see a name resolution error below, go to:")
print("   Kaggle Notebook Settings (⚙ right panel) → Internet → Enable")



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
faster-whisper installed ✅

⚠️  If you see a name resolution error below, go to:
   Kaggle Notebook Settings (⚙ right panel) → Internet → Enable


## Step 2 — Generate Test Audio

In [2]:
# ── Audio source ────────────────────────────────────────────────────────────
# Option A (Kaggle with Internet ON): generate audio via gTTS
# Option B (No internet): upload an .mp3/.wav file to your Kaggle dataset
#            and set AUDIO_FILE to its path

import os
AUDIO_FILE = "test_meeting.mp3"  # change this if you uploaded your own file

if not os.path.exists(AUDIO_FILE):
    # Try gTTS (requires Internet ON in Kaggle settings)
    try:
        from gtts import gTTS
        meeting_text = (
            "Good morning team. Let's start our daily standup. "
            "Ahmed will finish the login module by Wednesday. "
            "Sara is blocked by missing design specs. "
            "We decided to deploy to staging on Monday. "
            "Main blocker is the slow CI pipeline."
        )
        tts = gTTS(text=meeting_text, lang='en')
        tts.save(AUDIO_FILE)
        print(f"Audio generated via gTTS: {AUDIO_FILE}")
    except Exception as e:
        # Create a minimal silent WAV as placeholder
        import wave, struct, math
        with wave.open(AUDIO_FILE.replace('.mp3', '.wav'), 'w') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(16000)
            samples = [int(32767 * math.sin(2 * math.pi * 440 * t / 16000)) for t in range(16000 * 5)]
            wf.writeframes(struct.pack('<' + 'h' * len(samples), *samples))
        AUDIO_FILE = AUDIO_FILE.replace('.mp3', '.wav')
        print(f"gTTS unavailable ({e}). Created silent WAV: {AUDIO_FILE}")
        print("TIP: Upload a real .mp3 meeting recording to your Kaggle dataset instead.")
else:
    print(f"Using existing audio file: {AUDIO_FILE}")


Using existing audio file: test_meeting.mp3


## Step 3 — Transcribe with faster-whisper

In [3]:
import time
from faster_whisper import WhisperModel

# Load model (base = good balance of speed/accuracy)
# On Kaggle GPU use: device='cuda', compute_type='float16'
# On CPU use:        device='cpu',  compute_type='int8'
import torch
device       = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
print(f"Running on: {device} ({compute_type})")

model = WhisperModel('base', device=device, compute_type=compute_type)
print("Model loaded.")

start = time.time()
segments, info = model.transcribe('test_meeting.mp3', beam_size=5)
segments = list(segments)  # force evaluation
elapsed  = time.time() - start

# Assemble full transcript
transcript = ' '.join(s.text.strip() for s in segments)
print(f"\nTranscript ({elapsed:.2f}s):")
print(transcript)
print(f"\nLanguage detected: {info.language} ({info.language_probability:.1%})")

Running on: cpu (int8)
Model loaded.

Transcript (1.98s):
Armored will prepare the project's slides by Friday. Sarah will send the budget report to the manager by Monday. The team needs to review the design mock-ups before the next meeting on Wednesday.

Language detected: en (99.9%)


## Step 4 — Word-Level Timestamps

In [4]:
print("Segment timestamps:")
for seg in segments:
    print(f"  [{seg.start:.1f}s → {seg.end:.1f}s] {seg.text.strip()}")
    
audio_duration = segments[-1].end if segments else 1
rtf = elapsed / audio_duration
print(f"\nReal-Time Factor (RTF): {rtf:.2f}x")
print(f"  RTF < 1.0 = faster than real-time ✅" if rtf < 1 else f"  RTF > 1.0 = slower than real-time (CPU is expected)")

Segment timestamps:
  [0.0s → 4.0s] Armored will prepare the project's slides by Friday.
  [4.0s → 8.0s] Sarah will send the budget report to the manager by Monday.
  [8.0s → 14.0s] The team needs to review the design mock-ups before the next meeting on Wednesday.

Real-Time Factor (RTF): 0.14x
  RTF < 1.0 = faster than real-time ✅


## Step 5 — Save Transcript

In [5]:
import json
result = {
    "transcript": transcript,
    "language":   info.language,
    "confidence": info.language_probability,
    "rtf":        rtf,
    "segments":   [{"start": s.start, "end": s.end, "text": s.text} for s in segments]
}
with open('transcript_result.json', 'w') as f:
    json.dump(result, f, indent=2)
print("Saved: transcript_result.json")
print("\n✅ Whisper benchmark complete! This transcript format feeds directly into:")
print("   cognitive_service/transcriber.py → classifier.py → extractor.py")

Saved: transcript_result.json

✅ Whisper benchmark complete! This transcript format feeds directly into:
   cognitive_service/transcriber.py → classifier.py → extractor.py
